# Build donor-disjoint spot-overlap training and validation archives

This pipeline reads **all HDF5 experiments** in `data_esrf/all_friedelpairs`, reconstructs the stored spot patches and masks, transforms two spots independently, and combines them with a requested mask overlap of 10–90%.

Important design rules:

- Spots are paired only within the same source file. Even experiments containing the same material are never mixed.
- Source donors are split before augmentation: approximately 80% train and 20% validation.
- Both members of a Friedel pair always stay in the same split, preventing near-duplicate leakage. Friedel pairing is not otherwise used for augmentation.
- The final 100,000 samples are distributed nearly equally across the nine experiments.
- The output is directly compatible with `H5SpotSeparationDataset` in `train.py`: every sample contains `image`, `spot_images`, and `spot_masks`.
- Full generation is guarded by `RUN_FULL_GENERATION = False`; inspect the preview before enabling it.

In [ ]:
from contextlib import contextmanager
from pathlib import Path
import json
import zlib

import h5py
import hdf5plugin  # noqa: F401 - registers source-file compression filters
import matplotlib.pyplot as plt
import numpy as np


def find_project_dir():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "train.py").exists() and (candidate / "augment_data").exists():
            return candidate
    raise RuntimeError("Could not find the project directory containing train.py.")


PROJECT_DIR = find_project_dir()
SOURCE_DIR = PROJECT_DIR.parent / "data_esrf" / "all_friedelpairs"
TRAIN_FILE = PROJECT_DIR.parent / "data_esrf" / "augmented_spots_train.h5"
VALIDATION_FILE = PROJECT_DIR.parent / "data_esrf" / "augmented_spots_validation.h5"
PREVIEW_FILE = PROJECT_DIR / "augment_data" / "new_augmentation_preview.png"

TOTAL_SAMPLES = 10_000
VALIDATION_FRACTION = 0.20
FIXED_PATCH_SHAPE = (128, 128)
PATCH_MARGIN = 8
OVERLAP_RANGE = (0.10, 0.90)
ROTATION_RANGE = (0.0, 360.0)
ALLOW_FLIPS = True
LAYOUT_ATTEMPTS = 64
MAX_SAMPLE_ATTEMPTS = 64
RNG_SEED = 7
OVERWRITE = False
RUN_FULL_GENERATION = False

SOURCE_FILES = sorted(SOURCE_DIR.glob("*.h5"))
if not SOURCE_FILES:
    raise FileNotFoundError(f"No HDF5 files found in {SOURCE_DIR}")

print(f"Sources:    {SOURCE_DIR}")
print(f"Train:      {TRAIN_FILE}")
print(f"Validation: {VALIDATION_FILE}")
print(f"Experiments ({len(SOURCE_FILES)}):")
for source in SOURCE_FILES:
    print("  ", source.name)

## 1. Read spot tables and build leakage-safe donor splits

Most files store spots at `im_seg/spots`; `Al_segvol.h5` stores the identical table layout at `segvol/spots`. A donor group is one symmetric Friedel pair (or a singleton if a partner is unavailable). Groups—not individual rows—are assigned to train or validation.

In [ ]:
def find_spot_group(h5_file):
    for candidate in ("im_seg/spots", "segvol/spots"):
        if candidate in h5_file:
            return candidate
    raise KeyError("Neither im_seg/spots nor segvol/spots exists")


class SpotArchive:
    def __init__(self, path, seed=RNG_SEED, validation_fraction=VALIDATION_FRACTION):
        self.path = Path(path)
        self.name = self.path.stem
        self._h5 = None

        with h5py.File(self.path, "r") as f:
            self.group_path = find_spot_group(f)
            group = f[self.group_path]
            self.blob_id = group["blob_id"][:]
            self.frame_idx = group["frame_idx"][:]
            self.partner_blob_id = group["friedel_partner_id"][:]
            self.offset = group["offset"][:]
            self.shape = group["shape"][:]
            self.ncc_score = group["ncc_score"][:]

        self.n_spots = len(self.blob_id)
        self.area = self.shape[:, 0].astype(np.int64) * self.shape[:, 1].astype(np.int64)
        lookup = {int(blob): row for row, blob in enumerate(self.blob_id)}
        self.partner_row = np.array(
            [lookup.get(int(partner), -1) for partner in self.partner_blob_id], dtype=np.int64
        )

        # Canonical group ID: the smaller row of a resolved pair, otherwise this row.
        rows = np.arange(self.n_spots, dtype=np.int64)
        resolved = self.partner_row >= 0
        self.donor_group = rows.copy()
        self.donor_group[resolved] = np.minimum(rows[resolved], self.partner_row[resolved])
        groups = np.unique(self.donor_group)

        stable_seed = int(seed) + zlib.crc32(self.name.encode("utf-8"))
        rng = np.random.default_rng(stable_seed)
        shuffled_groups = groups.copy()
        rng.shuffle(shuffled_groups)
        n_validation = int(round(len(groups) * validation_fraction))
        n_validation = min(max(n_validation, 1), len(groups) - 1)
        validation_groups = set(map(int, shuffled_groups[:n_validation]))
        is_validation = np.fromiter(
            (int(group) in validation_groups for group in self.donor_group),
            dtype=bool,
            count=self.n_spots,
        )
        self.rows = {
            "train": rows[~is_validation],
            "validation": rows[is_validation],
        }
        self.groups = {
            split: np.unique(self.donor_group[split_rows])
            for split, split_rows in self.rows.items()
        }

    def open(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.path, "r")
        return self

    def close(self):
        if self._h5 is not None:
            self._h5.close()
            self._h5 = None

    def read_spot(self, row, margin=PATCH_MARGIN):
        self.open()
        row = int(row)
        height, width = map(int, self.shape[row])
        start = int(self.offset[row])
        stop = start + height * width
        group = self._h5[self.group_path]
        patch = group["patches"][start:stop].reshape(height, width).astype(np.float32)
        mask = group["masks"][start:stop].reshape(height, width).astype(bool)
        if not mask.any():
            raise ValueError(f"Empty mask in {self.name}, row {row}")

        background = patch[~mask]
        local_background = float(np.median(background)) if background.size else 0.0
        signal = patch - local_background
        signal[~mask] = 0.0
        signal[~np.isfinite(signal)] = 0.0
        np.maximum(signal, 0.0, out=signal)

        rr, cc = tight_slices(mask, margin=margin)
        return {
            "experiment": self.name,
            "source_file": self.path.name,
            "row": row,
            "blob_id": int(self.blob_id[row]),
            "frame": int(self.frame_idx[row]),
            "donor_group": int(self.donor_group[row]),
            "ncc": float(self.ncc_score[row]),
            "image": np.ascontiguousarray(signal[rr, cc]),
            "mask": np.ascontiguousarray(mask[rr, cc]),
        }


@contextmanager
def opened_archives(archives):
    try:
        for archive in archives:
            archive.open()
        yield archives
    finally:
        for archive in archives:
            archive.close()


ARCHIVES = [SpotArchive(path) for path in SOURCE_FILES]

print(f"{'experiment':38s} {'spots':>9s} {'groups':>9s} {'train rows':>11s} {'val rows':>9s}")
for archive in ARCHIVES:
    print(
        f"{archive.name:38s} {archive.n_spots:9,d} {len(np.unique(archive.donor_group)):9,d} "
        f"{len(archive.rows['train']):11,d} {len(archive.rows['validation']):9,d}"
    )

# Hard leakage checks at source-row and Friedel-group level.
for archive in ARCHIVES:
    assert not np.intersect1d(archive.rows["train"], archive.rows["validation"]).size
    assert not np.intersect1d(archive.groups["train"], archive.groups["validation"]).size
    assert len(archive.rows["train"]) + len(archive.rows["validation"]) == archive.n_spots
print("\nAll source splits are donor-disjoint and keep Friedel partners together.")

## 2. Transformation and overlap helpers

Each donor is background-subtracted, masked, randomly flipped, and rotated by an arbitrary angle. Bilinear interpolation is used for intensity and nearest-neighbour interpolation for the binary mask. Samples that cannot fit completely inside the fixed canvas are retried rather than cropped.

In [ ]:
def tight_slices(mask, margin=0):
    rows, cols = np.where(mask)
    if rows.size == 0:
        raise ValueError("Cannot crop an empty mask")
    r0 = max(int(rows.min()) - margin, 0)
    c0 = max(int(cols.min()) - margin, 0)
    r1 = min(int(rows.max()) + margin + 1, mask.shape[0])
    c1 = min(int(cols.max()) + margin + 1, mask.shape[1])
    return slice(r0, r1), slice(c0, c1)


def rotate_image_and_mask(image, mask, angle_degrees):
    angle = np.deg2rad(angle_degrees)
    cos_a, sin_a = float(np.cos(angle)), float(np.sin(angle))
    height, width = image.shape
    corners = np.array(
        [[0, 0], [0, width - 1], [height - 1, 0], [height - 1, width - 1]],
        dtype=np.float32,
    )
    center = np.array([(height - 1) / 2, (width - 1) / 2], dtype=np.float32)
    relative = corners - center
    rotated = np.column_stack(
        [
            relative[:, 0] * cos_a - relative[:, 1] * sin_a,
            relative[:, 0] * sin_a + relative[:, 1] * cos_a,
        ]
    )
    minimum = np.floor(rotated.min(axis=0))
    maximum = np.ceil(rotated.max(axis=0))
    out_height, out_width = (maximum - minimum + 1).astype(int)

    out_rows, out_cols = np.indices((out_height, out_width), dtype=np.float32)
    relative_rows = out_rows + minimum[0]
    relative_cols = out_cols + minimum[1]
    source_rows = relative_rows * cos_a + relative_cols * sin_a + center[0]
    source_cols = -relative_rows * sin_a + relative_cols * cos_a + center[1]

    nearest_rows = np.rint(source_rows).astype(np.int32)
    nearest_cols = np.rint(source_cols).astype(np.int32)
    nearest_valid = (
        (nearest_rows >= 0) & (nearest_rows < height)
        & (nearest_cols >= 0) & (nearest_cols < width)
    )
    rotated_mask = np.zeros((out_height, out_width), dtype=bool)
    rotated_mask[nearest_valid] = mask[nearest_rows[nearest_valid], nearest_cols[nearest_valid]]

    r0 = np.floor(source_rows).astype(np.int32)
    c0 = np.floor(source_cols).astype(np.int32)
    r1, c1 = r0 + 1, c0 + 1
    linear_valid = (r0 >= 0) & (r1 < height) & (c0 >= 0) & (c1 < width)
    dr, dc = source_rows - r0, source_cols - c0
    rotated_image = np.zeros((out_height, out_width), dtype=np.float32)
    rotated_image[linear_valid] = (
        image[r0[linear_valid], c0[linear_valid]] * (1 - dr[linear_valid]) * (1 - dc[linear_valid])
        + image[r1[linear_valid], c0[linear_valid]] * dr[linear_valid] * (1 - dc[linear_valid])
        + image[r0[linear_valid], c1[linear_valid]] * (1 - dr[linear_valid]) * dc[linear_valid]
        + image[r1[linear_valid], c1[linear_valid]] * dr[linear_valid] * dc[linear_valid]
    )
    return rotated_image, rotated_mask


def random_variant(spot, rng):
    image, mask = spot["image"], spot["mask"]
    flip_ud = bool(ALLOW_FLIPS and rng.integers(0, 2))
    flip_lr = bool(ALLOW_FLIPS and rng.integers(0, 2))
    angle = float(rng.uniform(*ROTATION_RANGE))
    if flip_ud:
        image, mask = np.flipud(image), np.flipud(mask)
    if flip_lr:
        image, mask = np.fliplr(image), np.fliplr(mask)

    image, mask = rotate_image_and_mask(image, mask, angle)
    if not mask.any():
        raise ValueError("Rotation produced an empty mask")
    rr, cc = tight_slices(mask)
    image = np.ascontiguousarray(image[rr, cc], dtype=np.float32)
    mask = np.ascontiguousarray(mask[rr, cc], dtype=bool)
    image[~mask] = 0.0
    return {
        **{key: spot[key] for key in ("experiment", "source_file", "row", "blob_id", "frame", "donor_group", "ncc")},
        "image": image,
        "mask": mask,
        "flip_ud": flip_ud,
        "flip_lr": flip_lr,
        "angle_degrees": angle,
    }


def overlap_for_offset(left_mask, right_mask, right_top, right_left):
    left_height, left_width = left_mask.shape
    right_height, right_width = right_mask.shape
    lr0, lc0 = max(0, right_top), max(0, right_left)
    lr1 = min(left_height, right_top + right_height)
    lc1 = min(left_width, right_left + right_width)
    if lr1 <= lr0 or lc1 <= lc0:
        return 0.0
    rr0, rc0 = lr0 - right_top, lc0 - right_left
    rr1, rc1 = rr0 + (lr1 - lr0), rc0 + (lc1 - lc0)
    overlap = np.logical_and(left_mask[lr0:lr1, lc0:lc1], right_mask[rr0:rr1, rc0:rc1]).sum()
    return float(overlap / max(min(left_mask.sum(), right_mask.sum()), 1))


def positive_layout(left_shape, right_shape, right_top, right_left):
    left_height, left_width = left_shape
    right_height, right_width = right_shape
    origin_top, origin_left = min(0, right_top), min(0, right_left)
    left_top, left_left = -origin_top, -origin_left
    right_top, right_left = right_top - origin_top, right_left - origin_left
    canvas_height = max(left_top + left_height, right_top + right_height)
    canvas_width = max(left_left + left_width, right_left + right_width)
    return left_top, left_left, right_top, right_left, canvas_height, canvas_width


def random_pair_layout(left_mask, right_mask, requested_overlap, rng):
    left_height, left_width = left_mask.shape
    right_height, right_width = right_mask.shape
    target_height, target_width = FIXED_PATCH_SHAPE
    requested_overlap = float(np.clip(requested_overlap, *OVERLAP_RANGE))

    overlap_height = max(1, int(round(min(left_height, right_height) * requested_overlap)))
    overlap_width = max(1, int(round(min(left_width, right_width) * requested_overlap)))
    candidates = [
        (left_height - overlap_height, left_width - overlap_width),
        (left_height - overlap_height, -(right_width - overlap_width)),
        (-(right_height - overlap_height), left_width - overlap_width),
        (-(right_height - overlap_height), -(right_width - overlap_width)),
        ((left_height - right_height) // 2, (left_width - right_width) // 2),
    ]
    for _ in range(LAYOUT_ATTEMPTS):
        candidates.append(
            (
                int(rng.integers(-right_height + 1, left_height)),
                int(rng.integers(-right_width + 1, left_width)),
            )
        )

    best = None
    for right_top, right_left in candidates:
        layout = positive_layout(left_mask.shape, right_mask.shape, right_top, right_left)
        if layout[4] > target_height or layout[5] > target_width:
            continue
        measured = overlap_for_offset(left_mask, right_mask, right_top, right_left)
        range_penalty = 0 if OVERLAP_RANGE[0] <= measured <= OVERLAP_RANGE[1] else 1
        score = (range_penalty, abs(measured - requested_overlap), -measured)
        if best is None or score < best[0]:
            best = (score, layout, measured)
    if best is None:
        raise ValueError("No complete pair layout fits the fixed canvas")
    return (*best[1], float(best[2]))


def paste_channel(channel, mask_channel, variant, top, left):
    height, width = variant["mask"].shape
    region = channel[top:top + height, left:left + width]
    mask_region = mask_channel[top:top + height, left:left + width]
    region[variant["mask"]] = variant["image"][variant["mask"]]
    mask_region[variant["mask"]] = 1
    return [int(top), int(left), int(top + height), int(left + width)]


def center_pad(array, target_shape=FIXED_PATCH_SHAPE):
    target_height, target_width = target_shape
    *leading, height, width = array.shape
    if height > target_height or width > target_width:
        raise ValueError(f"Array {array.shape} does not fit target {target_shape}")
    output = np.zeros((*leading, target_height, target_width), dtype=array.dtype)
    top, left = (target_height - height) // 2, (target_width - width) // 2
    output[..., top:top + height, left:left + width] = array
    return output, (top, left)

## 3. Generate one augmented sample

The two ground-truth intensity channels and masks retain their fixed identities. Their sum is the overlapped input image. A sample retries with different donors/transforms if the complete pair cannot fit the fixed canvas.

In [ ]:
def choose_two_donors(archive, split, rng):
    rows = archive.rows[split]
    if len(archive.groups[split]) < 2:
        raise ValueError(f"{archive.name}/{split} has fewer than two independent donor groups")
    for _ in range(100):
        left_row, right_row = map(int, rng.choice(rows, size=2, replace=False))
        if archive.donor_group[left_row] != archive.donor_group[right_row]:
            return left_row, right_row
    raise RuntimeError(f"Could not draw two independent donors from {archive.name}/{split}")


def make_overlap_sample(archive, split, rng):
    for _ in range(MAX_SAMPLE_ATTEMPTS):
        try:
            left_row, right_row = choose_two_donors(archive, split, rng)
            left = random_variant(archive.read_spot(left_row), rng)
            right = random_variant(archive.read_spot(right_row), rng)
            requested = float(rng.uniform(*OVERLAP_RANGE))
            layout = random_pair_layout(left["mask"], right["mask"], requested, rng)
            left_top, left_left, right_top, right_left, height, width, measured = layout

            spot_images = np.zeros((2, height, width), dtype=np.float32)
            spot_masks = np.zeros((2, height, width), dtype=np.uint8)
            left_bbox = paste_channel(spot_images[0], spot_masks[0], left, left_top, left_left)
            right_bbox = paste_channel(spot_images[1], spot_masks[1], right, right_top, right_left)
            spot_images, pad = center_pad(spot_images)
            spot_masks, _ = center_pad(spot_masks)
            image = spot_images.sum(axis=0, dtype=np.float32)

            donors = []
            for variant, bbox in ((left, left_bbox), (right, right_bbox)):
                donors.append({
                    "source_file": variant["source_file"],
                    "source_row": int(variant["row"]),
                    "blob_id": int(variant["blob_id"]),
                    "frame": int(variant["frame"]),
                    "donor_group": int(variant["donor_group"]),
                    "ncc": float(variant["ncc"]),
                    "flip_ud": bool(variant["flip_ud"]),
                    "flip_lr": bool(variant["flip_lr"]),
                    "angle_degrees": float(variant["angle_degrees"]),
                    "paste_bbox_before_padding": bbox,
                })
            metadata = {
                "experiment": archive.name,
                "split": split,
                "requested_overlap_fraction": requested,
                "measured_overlap_fraction": measured,
                "canvas_shape_before_padding": [int(height), int(width)],
                "padding_top_left": [int(pad[0]), int(pad[1])],
                "donors": donors,
            }
            return image, spot_images, spot_masks, metadata
        except (ValueError, RuntimeError):
            continue
    raise RuntimeError(f"Failed to make a fitting sample for {archive.name}/{split}")


def stable_rng(archive_name, split, seed=RNG_SEED):
    token = f"{archive_name}:{split}".encode("utf-8")
    return np.random.default_rng(int(seed) + zlib.crc32(token))

## 4. Preview every experiment before writing 100,000 samples

In [ ]:
def robust_limits(array, lower=1, upper=99.8):
    values = np.asarray(array)
    values = values[np.isfinite(values) & (values > 0)]
    if values.size == 0:
        return 0.0, 1.0
    low, high = np.percentile(values, [lower, upper])
    return float(low), float(max(high, low + 1e-6))


preview_samples = []
with opened_archives(ARCHIVES):
    for archive in ARCHIVES:
        rng = stable_rng(archive.name, "train")
        preview_samples.append((archive.name, make_overlap_sample(archive, "train", rng)))

fig, axes = plt.subplots(len(preview_samples), 5, figsize=(15, 3 * len(preview_samples)), squeeze=False)
for row, (experiment, sample) in enumerate(preview_samples):
    image, spot_images, spot_masks, metadata = sample
    panels = [image, spot_images[0], spot_masks[0], spot_images[1], spot_masks[1]]
    titles = [
        f"{experiment}\ninput | overlap={metadata['measured_overlap_fraction']:.2f}",
        "spot intensity 1", "spot mask 1", "spot intensity 2", "spot mask 2",
    ]
    for col, (panel, title) in enumerate(zip(panels, titles)):
        if col in (2, 4):
            axes[row, col].imshow(panel, cmap="gray", vmin=0, vmax=1)
        else:
            low, high = robust_limits(panel)
            axes[row, col].imshow(panel, cmap="gray", vmin=low, vmax=high)
        axes[row, col].set_title(title, fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
fig.savefig(PREVIEW_FILE, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved preview to {PREVIEW_FILE}")

## 5. Allocate an equal experiment ratio

The 80,000 training and 20,000 validation samples are allocated separately, so the global split is exact and each experiment differs from another by at most one sample.

In [ ]:
def allocate_evenly(total, names):
    names = list(names)
    base, remainder = divmod(int(total), len(names))
    return {name: base + int(index < remainder) for index, name in enumerate(names)}


experiment_names = [archive.name for archive in ARCHIVES]
validation_total = int(round(TOTAL_SAMPLES * VALIDATION_FRACTION))
train_total = TOTAL_SAMPLES - validation_total
TRAIN_QUOTAS = allocate_evenly(train_total, experiment_names)
VALIDATION_QUOTAS = allocate_evenly(validation_total, experiment_names)

print(f"Training total:   {sum(TRAIN_QUOTAS.values()):,}")
print(f"Validation total: {sum(VALIDATION_QUOTAS.values()):,}")
print(f"Combined total:   {sum(TRAIN_QUOTAS.values()) + sum(VALIDATION_QUOTAS.values()):,}\n")
print(f"{'experiment':38s} {'train':>8s} {'validation':>11s} {'combined':>9s}")
for name in experiment_names:
    print(
        f"{name:38s} {TRAIN_QUOTAS[name]:8,d} {VALIDATION_QUOTAS[name]:11,d} "
        f"{TRAIN_QUOTAS[name] + VALIDATION_QUOTAS[name]:9,d}"
    )

## 6. Atomic archive writer

Generation writes temporary files first and renames them only after both splits finish successfully. Root-level provenance arrays make donor leakage auditable without opening 100,000 sample groups.

In [ ]:
def initialize_output(h5_file, split, quotas):
    total = sum(quotas.values())
    h5_file.attrs.update({
        "format": "two_spot_intensity_separation",
        "split": split,
        "seed": RNG_SEED,
        "fixed_patch_shape": json.dumps(list(FIXED_PATCH_SHAPE)),
        "overlap_range": json.dumps(list(OVERLAP_RANGE)),
        "rotation_range_degrees": json.dumps(list(ROTATION_RANGE)),
        "source_directory": str(SOURCE_DIR),
        "experiments": json.dumps(experiment_names),
        "sample_quotas": json.dumps(quotas),
        "friedel_split_policy": "both members kept in same source split; otherwise ignored",
    })
    provenance = h5_file.create_group("provenance")
    provenance.create_dataset("experiment_index", shape=(total,), dtype=np.int16)
    provenance.create_dataset("source_rows", shape=(total, 2), dtype=np.int32)
    provenance.create_dataset("source_blob_ids", shape=(total, 2), dtype=np.int32)
    provenance.create_dataset("source_donor_groups", shape=(total, 2), dtype=np.int32)
    provenance.create_dataset("requested_overlap", shape=(total,), dtype=np.float32)
    provenance.create_dataset("measured_overlap", shape=(total,), dtype=np.float32)

    source_splits = h5_file.create_group("source_splits")
    for archive in ARCHIVES:
        group = source_splits.create_group(archive.name)
        group.create_dataset("rows", data=archive.rows[split], compression="gzip")
        group.create_dataset("donor_groups", data=archive.groups[split], compression="gzip")
    return provenance


def write_split_to_temp(path, split, quotas):
    temporary = path.with_name(f"{path.stem}.tmp{path.suffix}")
    if temporary.exists():
        temporary.unlink()
    total = sum(quotas.values())
    sample_index = 0
    with h5py.File(temporary, "w") as output:
        provenance = initialize_output(output, split, quotas)
        with opened_archives(ARCHIVES):
            for experiment_index, archive in enumerate(ARCHIVES):
                rng = stable_rng(archive.name, split)
                quota = quotas[archive.name]
                print(f"{split:10s} | {archive.name:38s} | {quota:6,d} samples")
                for _ in range(quota):
                    image, spot_images, spot_masks, metadata = make_overlap_sample(archive, split, rng)
                    sample = output.create_group(f"sample_{sample_index:06d}")
                    sample.create_dataset("image", data=image, compression="gzip", compression_opts=4)
                    sample.create_dataset("spot_images", data=spot_images, compression="gzip", compression_opts=4)
                    sample.create_dataset("spot_masks", data=spot_masks, compression="gzip", compression_opts=4)
                    sample.attrs["experiment"] = archive.name
                    sample.attrs["split"] = split
                    sample.attrs["requested_overlap_fraction"] = metadata["requested_overlap_fraction"]
                    sample.attrs["measured_overlap_fraction"] = metadata["measured_overlap_fraction"]
                    sample.attrs["donors"] = json.dumps(metadata["donors"])

                    donors = metadata["donors"]
                    provenance["experiment_index"][sample_index] = experiment_index
                    provenance["source_rows"][sample_index] = [donor["source_row"] for donor in donors]
                    provenance["source_blob_ids"][sample_index] = [donor["blob_id"] for donor in donors]
                    provenance["source_donor_groups"][sample_index] = [donor["donor_group"] for donor in donors]
                    provenance["requested_overlap"][sample_index] = metadata["requested_overlap_fraction"]
                    provenance["measured_overlap"][sample_index] = metadata["measured_overlap_fraction"]
                    sample_index += 1
                    if sample_index % 1000 == 0:
                        output.flush()
                        print(f"  wrote {sample_index:,}/{total:,}")
    assert sample_index == total
    return temporary


def write_augmented_archives(overwrite=OVERWRITE):
    for path in (TRAIN_FILE, VALIDATION_FILE):
        if path.exists() and not overwrite:
            raise FileExistsError(f"{path} exists. Set OVERWRITE=True to replace it.")
    train_temp = validation_temp = None
    try:
        train_temp = write_split_to_temp(TRAIN_FILE, "train", TRAIN_QUOTAS)
        validation_temp = write_split_to_temp(VALIDATION_FILE, "validation", VALIDATION_QUOTAS)
        train_temp.replace(TRAIN_FILE)
        validation_temp.replace(VALIDATION_FILE)
    except Exception:
        for temporary in (train_temp, validation_temp):
            if temporary is not None and temporary.exists():
                temporary.unlink()
        raise
    print(f"Wrote {TRAIN_FILE}")
    print(f"Wrote {VALIDATION_FILE}")

## 7. Validate structure, ratios, targets, and donor isolation

In [ ]:
def sample_group_names(h5_file):
    return sorted(
        name for name, obj in h5_file.items()
        if isinstance(obj, h5py.Group)
        and "image" in obj and "spot_images" in obj and "spot_masks" in obj
    )


def validate_augmented_archives(train_file=TRAIN_FILE, validation_file=VALIDATION_FILE, checks_per_file=20):
    for path in (train_file, validation_file):
        if not Path(path).exists():
            print(f"Not generated yet: {path}")
            return False

    donor_sets = {}
    for split, path in (("train", train_file), ("validation", validation_file)):
        with h5py.File(path, "r") as f:
            names = sample_group_names(f)
            provenance = f["provenance"]
            experiments = provenance["experiment_index"][:]
            rows = provenance["source_rows"][:]
            assert len(names) == len(rows)
            assert np.all(rows[:, 0] != rows[:, 1])
            keys = {
                (int(experiment), int(row))
                for experiment, pair in zip(experiments, rows)
                for row in pair
            }
            donor_sets[split] = keys

            counts = np.bincount(experiments, minlength=len(experiment_names))
            print(f"\n{split}: {len(names):,} samples")
            for name, count in zip(experiment_names, counts):
                print(f"  {name:38s}: {int(count):,}")

            indices = np.linspace(0, len(names) - 1, min(checks_per_file, len(names)), dtype=int)
            for index in indices:
                group = f[names[index]]
                image = group["image"][:]
                spot_images = group["spot_images"][:]
                spot_masks = group["spot_masks"][:]
                assert image.shape == FIXED_PATCH_SHAPE
                assert spot_images.shape == (2, *FIXED_PATCH_SHAPE)
                assert spot_masks.shape == (2, *FIXED_PATCH_SHAPE)
                assert np.allclose(image, spot_images.sum(axis=0), rtol=1e-5, atol=1e-5)
                assert set(np.unique(spot_masks)).issubset({0, 1})
                assert np.all(spot_images[spot_masks == 0] == 0)

    leakage = donor_sets["train"] & donor_sets["validation"]
    assert not leakage, f"Found {len(leakage)} source donors in both files"
    print("\nValidation passed: shapes, target sums, masks, experiment ratios, and donor isolation are correct.")
    return True

## 8. Full generation

Review `new_augmentation_preview.png`, confirm available disk space, then set `RUN_FULL_GENERATION = True`. Generation is intentionally not started merely by opening the notebook.

In [ ]:
if RUN_FULL_GENERATION:
    write_augmented_archives(overwrite=OVERWRITE)
    validate_augmented_archives()
else:
    print("Preview complete. Set RUN_FULL_GENERATION = True to create the 100,000-sample archives.")